In [1]:
import requests
import pandas as pd
import zipfile
import os
import time

from pathlib import Path

In [2]:
BASE_URL = "https://api.insee.fr/melodi"
TIMEOUT = 30

response = requests.get(
    f"{BASE_URL}/catalog/all",
    params={"lang": "fr"},
    timeout=TIMEOUT,
)
response.raise_for_status()
print(response.url)

catalog = response.json()
type(catalog)

https://api.insee.fr/melodi/catalog/all?lang=fr


list

In [3]:
def get_text(items, lang="fr"):
    if not items:
        return None

    for item in items:
        if item.get("lang") == lang:
            return item.get("content")

    return items[0].get("content")


catalog_rows = []

for dataset in catalog:
    spatial_levels = [
        item.get("id")
        for item in dataset.get("spatialResolution", [])
    ]

    catalog_rows.append(
        {
            "identifier": dataset.get("identifier"),
            "title": get_text(dataset.get("title")),
            "abstract": get_text(dataset.get("abstract")),
            "description": get_text(dataset.get("description")),
            "start_period": dataset.get("temporal", {}).get("startPeriod"),
            "end_period": dataset.get("temporal", {}).get("endPeriod"),
            "spatial_levels": spatial_levels,
            "num_observations": dataset.get("numObservations"),
            "modified": dataset.get("modified"),
        }
    )

catalog_df = pd.DataFrame(catalog_rows)
catalog_df.head()

,identifier,title,abstract,description,start_period,end_period,spatial_levels,num_observations,modified
0,DD_CNA_AGREGATS,Produit Intérieur Brut (PIB) et grands agrégat...,Données annuelles de Produit Intérieur Brut (P...,Le produit intérieur brut (PIB) est le princip...,1949-01-01T00:00:00,2025-01-01T00:00:00,[NAT],233234,2026-06-16T15:59:44.541846557
1,DD_CNA_APU,Comptes des administrations publiques,Données concernant le secteur institutionnel d...,Le secteur institutionnel des administrations ...,1949-01-01T00:00:00,2025-01-01T00:00:00,[NAT],620934,2026-06-08T10:17:36.514543289
2,DD_CNA_BRANCHES,Activité des branches de l'économie,Données annuelles sur l'activité des branches ...,Données annuelles sur l'activité des branches ...,1949-01-01T00:00:00,2025-01-01T00:00:00,[NAT],437533,2026-05-29T08:56:36.474968668
3,DD_CNA_CONSO_MENAGES_COICOP,Consommation des ménages par fonction,Données annuelles de consommation finale effec...,Données annuelles de consommation finale effec...,1949-01-01T00:00:00,2025-01-01T00:00:00,[NAT],260261,2026-05-29T09:03:36.396354644
4,DD_CNA_CONSO_MENAGES_PRODUITS,Consommation des ménages par produit,Données annuelles de consommation finale effec...,Données annuelles de consommation finale effec...,1949-01-01T00:00:00,2025-01-01T00:00:00,[NAT],1044160,2026-05-29T09:00:36.607139214


In [4]:
local_catalog = catalog_df[
    catalog_df["spatial_levels"].apply(
        # lambda levels: bool({"DEP", "ARR", "ARM"} & set(levels))
        lambda levels: bool({"DEP"} & set(levels))
    )
].copy()

len(local_catalog)

101

In [5]:
def search_catalog(term):
    searchable = (
        local_catalog["title"].fillna("")
        + " "
        + local_catalog["abstract"].fillna("")
        + " "
        + local_catalog["description"].fillna("")
    )

    return local_catalog[
        searchable.str.contains(term, case=False, regex=False)
    ][
        [
            "identifier",
            "title",
            "start_period",
            "end_period",
            "spatial_levels",
            "num_observations",
        ]
    ]

In [ ]:
def search_full_catalog(term):
    searchable = (
        catalog_df["title"].fillna("")
        + " "
        + catalog_df["abstract"].fillna("")
        + " "
        + catalog_df["description"].fillna("")
    )

    return catalog_df.loc[
        searchable.str.contains(
            term,
            case=False,
            regex=False,
        ),
        [
            "identifier",
            "title",
            "abstract",
            "start_period",
            "end_period",
            "spatial_levels",
            "num_observations",
        ],
    ]

These are the metrics we are interested in exploring

```python
search_catalog("population")
search_catalog("chômage")
search_catalog("pauvreté")
search_catalog("salaire")
search_catalog("densité")
```

## Access functions

In [6]:
def get_dataset_catalog(dataset_id, lang="fr"):
    response = requests.get(
        f"{BASE_URL}/catalog/{dataset_id}",
        params={"lang": lang},
        timeout=TIMEOUT,
    )
    response.raise_for_status()
    return response.json()

In [7]:
def latest_department_slice(
    df: pd.DataFrame,
    *,
    optional_totals: tuple[str, ...] = ("SEX", "AGE"),
    extra_filters: dict[str, str] | None = None,
) -> pd.DataFrame:
    required = {
        "GEO",
        "GEO_OBJECT",
        "TIME_PERIOD",
        "OBS_VALUE",
    }

    missing = required - set(df.columns)
    if missing:
        raise ValueError(
            f"Missing required columns: {sorted(missing)}"
        )

    mask = df["GEO_OBJECT"].eq("DEP")

    # Some datasets expose frequency explicitly; others do not.
    if "FREQ" in df.columns:
        available_frequencies = set(df["FREQ"].dropna().unique())

        if "A" not in available_frequencies:
            raise ValueError(
                "FREQ exists, but annual frequency 'A' is unavailable. "
                f"Available values: {sorted(available_frequencies)}"
            )

        mask &= df["FREQ"].eq("A")

    # Apply total-category filters only when the dimensions exist.
    for dimension in optional_totals:
        if dimension in df.columns:
            available_values = set(df[dimension].dropna().unique())

            if "_T" not in available_values:
                raise ValueError(
                    f"{dimension} exists, but total code '_T' is unavailable. "
                    f"Available values: {sorted(available_values)}"
                )

            mask &= df[dimension].eq("_T")

    # Dataset-specific dimensions, such as measure or unit.
    for column, value in (extra_filters or {}).items():
        if column not in df.columns:
            raise ValueError(
                f"Requested filter column {column!r} is absent."
            )

        available_values = set(df[column].dropna().unique())
        if value not in available_values:
            raise ValueError(
                f"Requested {column}={value!r} is unavailable. "
                f"Available values: {sorted(available_values)}"
            )

        mask &= df[column].eq(value)

    filtered = df.loc[mask].copy()

    if filtered.empty:
        raise ValueError(
            "No departmental observations remain after filtering."
        )

    filtered["TIME_PERIOD"] = pd.to_numeric(
        filtered["TIME_PERIOD"],
        errors="raise",
    )

    latest_year = filtered["TIME_PERIOD"].max()

    result = (
        filtered.loc[filtered["TIME_PERIOD"].eq(latest_year)]
        .sort_values("GEO", kind="stable")
        .reset_index(drop=True)
    )

    return result

## Salary data

In [9]:
salary_candidates = search_catalog("salaire")
salary_candidates

,identifier,title,start_period,end_period,spatial_levels,num_observations
18,DS_BTS_SAL_EQTP_SEX_AGE,Salaires dans le secteur privé selon le sexe e...,2022-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",445248
19,DS_BTS_SAL_EQTP_SEX_PCS,Salaires dans le secteur privé selon le sexe e...,2022-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",371040
25,DS_COMPTES_REGIONAUX,Comptes régionaux annuels,1990-01-01T00:00:00,2024-01-01T00:00:00,"[FRANCE, OTHER, DEP, REG]",196827


In [10]:
salary_metadata = {
    dataset_id: get_dataset_catalog(dataset_id)
    for dataset_id in salary_candidates["identifier"]
}

salary_metadata.keys()

dict_keys(['DS_BTS_SAL_EQTP_SEX_AGE', 'DS_BTS_SAL_EQTP_SEX_PCS', 'DS_COMPTES_REGIONAUX'])

In [11]:
dataset_id = "DS_BTS_SAL_EQTP_SEX_AGE"

metadata = salary_metadata[dataset_id]

csv_product = next(
    product
    for product in metadata["product"]
    if product.get("format") == "CSV"
)

csv_url = csv_product["accessURL"]
csv_url

'https://api.insee.fr/melodi/file/DS_BTS_SAL_EQTP_SEX_AGE/DS_BTS_SAL_EQTP_SEX_AGE_2023_CSV_FR'

In [12]:
discovery_dir = Path("tmp/demographics_discovery")
discovery_dir.mkdir(parents=True, exist_ok=True)

download_path = discovery_dir / f"{dataset_id}.zip"

response = requests.get(csv_url, timeout=TIMEOUT)
response.raise_for_status()

download_path.write_bytes(response.content)
download_path

PosixPath('tmp/demographics_discovery/DS_BTS_SAL_EQTP_SEX_AGE.zip')

In [13]:
with zipfile.ZipFile(download_path) as archive:
    print(archive.namelist())

['DS_BTS_SAL_EQTP_SEX_AGE_2023_metadata.csv', 'DS_BTS_SAL_EQTP_SEX_AGE_2023_data.csv']


In [14]:
with zipfile.ZipFile(download_path) as archive:
    csv_names = [
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]

    data_names = [
        name for name in csv_names
        if name.lower().endswith("_data.csv")
    ]
    metadata_names = [
        name for name in csv_names
        if name.lower().endswith("_metadata.csv")
    ]

    if len(data_names) != 1:
        raise ValueError(f"Expected one data CSV, found: {data_names}")
    if len(metadata_names) != 1:
        raise ValueError(f"Expected one metadata CSV, found: {metadata_names}")

    with archive.open(data_names[0]) as file:
        department_chunks = []

        for chunk in pd.read_csv(
            file,
            sep=";",
            dtype=str,
            chunksize=50_000,
            low_memory=False,
        ):
            department_chunks.append(
                chunk.loc[chunk["GEO_OBJECT"].eq("DEP")].copy()
            )

        wage_raw = pd.concat(department_chunks, ignore_index=True)

    with archive.open(metadata_names[0]) as file:
        wage_metadata = pd.read_csv(file, sep=";")

print("Data:", data_names[0], wage_raw.shape)
print("Geographic levels retained:", wage_raw["GEO_OBJECT"].unique().tolist())
print("Metadata:", metadata_names[0], wage_metadata.shape)

Data: DS_BTS_SAL_EQTP_SEX_AGE_2023_data.csv (3600, 9)
Geographic levels retained: ['DEP']
Metadata: DS_BTS_SAL_EQTP_SEX_AGE_2023_metadata.csv (12372, 4)


In [15]:
latest_year = wage_raw["TIME_PERIOD"].astype(int).max()

wage_total = (
    wage_raw.loc[
        wage_raw["AGE"].eq("_T")
        & wage_raw["SEX"].eq("_T")
        & wage_raw["TIME_PERIOD"].astype(int).eq(latest_year)
    ]
    .sort_values("GEO", kind="stable")
    .reset_index(drop=True)
)

print("Latest year:", latest_year)
print("Rows with AGE == '_T' and SEX == '_T':", len(wage_total))
print("Departments:", wage_total["GEO"].nunique())
wage_total.head()

Latest year: 2023
Rows with AGE == '_T' and SEX == '_T': 100
Departments: 100


,GEO,GEO_OBJECT,FREQ,SEX,AGE,DERA_MEASURE,CONF_STATUS,TIME_PERIOD,OBS_VALUE
0,01,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2489.861901
1,02,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2287.448625
2,03,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2249.54134
3,04,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2217.440392
4,05,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2189.127665


In [16]:
wage_raw.columns.tolist()

['GEO',
 'GEO_OBJECT',
 'FREQ',
 'SEX',
 'AGE',
 'DERA_MEASURE',
 'CONF_STATUS',
 'TIME_PERIOD',
 'OBS_VALUE']

In [17]:
wage_raw.head()

,GEO,GEO_OBJECT,FREQ,SEX,AGE,DERA_MEASURE,CONF_STATUS,TIME_PERIOD,OBS_VALUE
0,56,DEP,A,_T,Y_GE55,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2022,2427.433135
1,67,DEP,A,_T,Y50T54,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2874.415625
2,67,DEP,A,F,Y25T39,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2022,2078.540273
3,974,DEP,A,_T,Y_GE55,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2646.945094
4,974,DEP,A,F,Y25T39,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2022,1995.553797


In [18]:
wage_metadata.columns.tolist()

['COD_VAR', 'LIB_VAR', 'COD_MOD', 'LIB_MOD']

In [19]:
wage_metadata.head(50)

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD
0,AGE,Âge,Y_LT25,Moins de 25 ans
1,AGE,Âge,Y25T39,De 25 à 39 ans
2,AGE,Âge,Y40T49,De 40 à 49 ans
3,AGE,Âge,Y50T54,De 50 à 54 ans
4,AGE,Âge,Y_GE55,55 ans ou plus
5,AGE,Âge,_T,Total
6,CONF_STATUS,Statut de confidentialité,F,Diffusable
7,CONF_STATUS,Statut de confidentialité,C,Confidentiel (secret statistique)
8,DERA_MEASURE,Mesure des salaires,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,Salaire net EQTP mensuel moyen
9,FREQ,Fréquence,A,Annuel


In [20]:
wage_total = latest_department_slice(wage_raw)

print("Latest year:", wage_total["TIME_PERIOD"].unique())
print("Rows:", len(wage_total))
print("Departments:", wage_total["GEO"].nunique())

if "CONF_STATUS" in wage_total.columns:
    print(wage_total["CONF_STATUS"].value_counts(dropna=False))

print("Missing values:", wage_total["OBS_VALUE"].isna().sum())

Latest year: [2023]
Rows: 100
Departments: 100
CONF_STATUS
F    100
Name: count, dtype: int64
Missing values: 0


In [21]:
wage_total

,GEO,GEO_OBJECT,FREQ,SEX,AGE,DERA_MEASURE,CONF_STATUS,TIME_PERIOD,OBS_VALUE
0,01,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2489.861901
1,02,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2287.448625
2,03,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2249.54134
3,04,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2217.440392
4,05,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2189.127665
...,...,...,...,...,...,...,...,...,...
95,95,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2642.744878
96,971,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2406.402715
97,972,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2416.911243
98,973,DEP,A,_T,_T,SALAIRE_NET_EQTP_MENSUEL_MOYENNE,F,2023,2464.593251


## Poverty data

In [22]:
poverty_candidates = search_catalog("pauvreté")
poverty_candidates

,identifier,title,start_period,end_period,spatial_levels,num_observations
47,DS_FILOSOFI_CC,"Niveau de vie, taux de pauvreté, décomposition...",2023-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, OTHER, BV2022, COM, DEP, E...",1123173
48,DS_FILOSOFI_SAGE_LOG_TP_NIVVIE,Niveau de vie médian et taux de pauvreté par t...,2023-01-01T00:00:00,2023-01-01T00:00:00,"[DEP, FRANCE, REG]",2688


In [23]:
poverty_metadata = {
    dataset_id: get_dataset_catalog(dataset_id)
    for dataset_id in poverty_candidates["identifier"]
}

poverty_metadata.keys()

dict_keys(['DS_FILOSOFI_CC', 'DS_FILOSOFI_SAGE_LOG_TP_NIVVIE'])

In [24]:
dataset_id = "DS_FILOSOFI_CC"

metadata = poverty_metadata[dataset_id]

csv_product = next(
    product
    for product in metadata["product"]
    if product.get("format") == "CSV"
)

csv_url = csv_product["accessURL"]
csv_url

'https://api.insee.fr/melodi/file/DS_FILOSOFI_CC/DS_FILOSOFI_CC_2023_CSV_FR'

In [25]:
discovery_dir = Path("tmp/demographics_discovery")
discovery_dir.mkdir(parents=True, exist_ok=True)

download_path = discovery_dir / f"{dataset_id}.zip"

response = requests.get(csv_url, timeout=TIMEOUT)
response.raise_for_status()

download_path.write_bytes(response.content)
download_path

PosixPath('tmp/demographics_discovery/DS_FILOSOFI_CC.zip')

In [26]:
with zipfile.ZipFile(download_path) as archive:
    print(archive.namelist())

['DS_FILOSOFI_CC_2023_metadata.csv', 'DS_FILOSOFI_CC_2023_data.csv']


In [27]:
with zipfile.ZipFile(download_path) as archive:
    csv_names = [
        name for name in archive.namelist()
        if name.lower().endswith(".csv")
    ]

    data_names = [
        name for name in csv_names
        if name.lower().endswith("_data.csv")
    ]
    metadata_names = [
        name for name in csv_names
        if name.lower().endswith("_metadata.csv")
    ]

    if len(data_names) != 1:
        raise ValueError(f"Expected one data CSV, found: {data_names}")
    if len(metadata_names) != 1:
        raise ValueError(f"Expected one metadata CSV, found: {metadata_names}")

    with archive.open(data_names[0]) as file:
        department_chunks = []

        for chunk in pd.read_csv(
            file,
            sep=";",
            dtype=str,
            chunksize=50_000,
            low_memory=False,
        ):
            department_chunks.append(
                chunk.loc[chunk["GEO_OBJECT"].eq("DEP")].copy()
            )

        poverty_raw = pd.concat(department_chunks, ignore_index=True)

    with archive.open(metadata_names[0]) as file:
        poverty_metadata = pd.read_csv(file, sep=";")

print("Data:", data_names[0], poverty_raw.shape)
print("Geographic levels retained:", poverty_raw["GEO_OBJECT"].unique().tolist())
print("Metadata:", metadata_names[0], poverty_metadata.shape)

Data: DS_FILOSOFI_CC_2023_data.csv (2619, 9)
Geographic levels retained: ['DEP']
Metadata: DS_FILOSOFI_CC_2023_metadata.csv (41647, 5)


In [28]:
poverty_raw.head()

,FILOSOFI_MEASURE,GEO,GEO_OBJECT,UNIT_MEASURE,CONF_STATUS,OBS_STATUS,UNIT_MULT,TIME_PERIOD,OBS_VALUE
0,S_DIR_TAX_DI,82,DEP,PT,F,A,0,2023,-13.8
1,D4_SL,01,DEP,EUR_YR,F,A,0,2023,25190
2,D1_SL,28,DEP,EUR_YR,F,A,0,2023,14190
3,D9_SL,2B,DEP,EUR_YR,F,A,0,2023,43630
4,S_EI_DI_UNE,28,DEP,PT,F,A,0,2023,2.2


In [29]:
poverty_metadata.head(10)

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,GEO_OBJECT
0,CONF_STATUS,Statut de confidentialité,F,Diffusable,NaN
1,CONF_STATUS,Statut de confidentialité,C,Confidentiel (secret statistique),NaN
2,FILOSOFI_MEASURE,Mesures filosofi,D1_SL,1er décile du niveau de vie (en euros),NaN
3,FILOSOFI_MEASURE,Mesures filosofi,D2_SL,2e décile du niveau de vie (en euros),NaN
4,FILOSOFI_MEASURE,Mesures filosofi,D3_SL,3e décile du niveau de vie (en euros),NaN
5,FILOSOFI_MEASURE,Mesures filosofi,D4_SL,4e décile du niveau de vie (en euros),NaN
6,FILOSOFI_MEASURE,Mesures filosofi,D6_SL,6e décile du niveau de vie (en euros),NaN
7,FILOSOFI_MEASURE,Mesures filosofi,D7_SL,7e décile du niveau de vie (en euros),NaN
8,FILOSOFI_MEASURE,Mesures filosofi,D8_SL,8e décile du niveau de vie (en euros),NaN
9,FILOSOFI_MEASURE,Mesures filosofi,D9_SL,9e décile du niveau de vie (en euros),NaN


In [30]:
def inspect_dimensions(df: pd.DataFrame) -> pd.DataFrame:
    rows = []

    for column in df.columns:
        values = df[column].drop_duplicates().tolist()

        rows.append(
            {
                "column": column,
                "n_unique": len(values),
                "sample_values": values[:25],
            }
        )

    return pd.DataFrame(rows)


inspect_dimensions(poverty_raw)

,column,n_unique,sample_values
0,FILOSOFI_MEASURE,27,"[S_DIR_TAX_DI, D4_SL, D1_SL, D9_SL, S_EI_DI_UN..."
1,GEO,97,"[82, 01, 28, 2B, 33, 23, 29, 32, 36, 31, 80, 0..."
2,GEO_OBJECT,1,[DEP]
3,UNIT_MEASURE,3,"[PT, EUR_YR, NR]"
4,CONF_STATUS,1,[F]
5,OBS_STATUS,1,[A]
6,UNIT_MULT,1,[0]
7,TIME_PERIOD,1,[2023]
8,OBS_VALUE,1436,"[-13.8, 25190, 14190, 43630, 2.2, 0.9, 28030, ..."


In [31]:
median_living_standard = latest_department_slice(
    poverty_raw,
    extra_filters={"FILOSOFI_MEASURE": "MED_SL"},
).rename(
    columns={"OBS_VALUE": "median_living_standard_eur"}
)

In [32]:
poverty_rate = latest_department_slice(
    poverty_raw,
    extra_filters={"FILOSOFI_MEASURE": "PR_MD60"},
).rename(
    columns={"OBS_VALUE": "poverty_rate_percent"}
)

In [33]:
for name, df in {
    "median_living_standard": median_living_standard,
    "poverty_rate": poverty_rate,
}.items():
    print(name)
    print("Latest year:", df["TIME_PERIOD"].unique())
    print("Rows:", len(df))
    print("Departments:", df["GEO"].nunique())
    if "CONF_STATUS" in df.columns:
        print(df["CONF_STATUS"].value_counts(dropna=False))

        value_column = (
            "median_living_standard_eur"
            if name == "median_living_standard"
            else "poverty_rate_percent"
        )
        print("Missing values:", df[value_column].isna().sum())
        print()

median_living_standard
Latest year: [2023]
Rows: 97
Departments: 97
CONF_STATUS
F    97
Name: count, dtype: int64
Missing values: 0

poverty_rate
Latest year: [2023]
Rows: 97
Departments: 97
CONF_STATUS
F    97
Name: count, dtype: int64
Missing values: 0



In [34]:
median_living_standard.head()

,FILOSOFI_MEASURE,GEO,GEO_OBJECT,UNIT_MEASURE,CONF_STATUS,OBS_STATUS,UNIT_MULT,TIME_PERIOD,median_living_standard_eur
0,MED_SL,01,DEP,EUR_YR,F,A,0,2023,27990
1,MED_SL,02,DEP,EUR_YR,F,A,0,2023,23360
2,MED_SL,03,DEP,EUR_YR,F,A,0,2023,24430
3,MED_SL,04,DEP,EUR_YR,F,A,0,2023,24240
4,MED_SL,05,DEP,EUR_YR,F,A,0,2023,25060


In [35]:
poverty_rate.head()

,FILOSOFI_MEASURE,GEO,GEO_OBJECT,UNIT_MEASURE,CONF_STATUS,OBS_STATUS,UNIT_MULT,TIME_PERIOD,poverty_rate_percent
0,PR_MD60,01,DEP,PT,F,A,0,2023,12.2
1,PR_MD60,02,DEP,PT,F,A,0,2023,19.4
2,PR_MD60,03,DEP,PT,F,A,0,2023,16.4
3,PR_MD60,04,DEP,PT,F,A,0,2023,18.6
4,PR_MD60,05,DEP,PT,F,A,0,2023,14.3


## Unemployment data

In [82]:
search_full_catalog("chômage")

,identifier,title,abstract,start_period,end_period,spatial_levels,num_observations
11,DD_EEC_ANNUEL,"Activité, emploi et chômage - résultats annuel...","Données annuelles sur la population totale, la...",2025-01-01T00:00:00,2025-01-01T00:00:00,[FRANCE],16398
12,DD_EEC_SERIES,"Activité, emploi et chômage - séries longues","Séries longues sur la population active, l’emp...",1975-01-01T00:00:00,2025-01-01T00:00:00,[FRANCE],75303
75,DS_RP_EMPLOI_LR_COMP,Population active et chômage - Princiapux indi...,Caractéristiques de la population active résid...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",2631447
76,DS_RP_EMPLOI_LR_PRINC,Population active et chômage - Principaux indi...,Caractéristiques de la population résidente de...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",16039296


In [61]:
unemployment_candidates = search_catalog("chômage")
unemployment_candidates

,identifier,title,start_period,end_period,spatial_levels,num_observations
75,DS_RP_EMPLOI_LR_COMP,Population active et chômage - Princiapux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",2631447
76,DS_RP_EMPLOI_LR_PRINC,Population active et chômage - Principaux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",16039296


In [48]:
unemployment_catalog_metadata = {
    dataset_id: get_dataset_catalog(dataset_id)
    for dataset_id in unemployment_candidates["identifier"]
}

unemployment_catalog_metadata.keys()

dict_keys(['DS_RP_EMPLOI_LR_COMP', 'DS_RP_EMPLOI_LR_PRINC'])

In [49]:
unemployment_dataset_id = "DS_RP_EMPLOI_LR_COMP"

unemployment_dataset_metadata = (
    unemployment_catalog_metadata[unemployment_dataset_id]
)

unemployment_csv_products = [
    product
    for product in unemployment_dataset_metadata.get("product", [])
    if product.get("format") == "CSV"
]

if len(unemployment_csv_products) != 1:
    display(pd.DataFrame(unemployment_csv_products))
    raise ValueError(
        "Expected exactly one CSV product for "
        f"{unemployment_dataset_id}."
    )

unemployment_csv_product = unemployment_csv_products[0]
unemployment_csv_url = unemployment_csv_product["accessURL"]
unemployment_csv_url

'https://api.insee.fr/melodi/file/DS_RP_EMPLOI_LR_COMP/DS_RP_EMPLOI_LR_COMP_2023_CSV_FR'

In [50]:
discovery_dir = Path("tmp/demographics_discovery")
discovery_dir.mkdir(parents=True, exist_ok=True)

unemployment_download_path = (
    discovery_dir / f"{unemployment_dataset_id}.zip"
)

unemployment_response = requests.get(
    unemployment_csv_url,
    timeout=(TIMEOUT, 300),
)

unemployment_response.raise_for_status()

unemployment_download_path.write_bytes(
    unemployment_response.content
)

print("Downloaded:", unemployment_download_path)
print(
    "Bytes:",
    unemployment_download_path.stat().st_size,
)

Downloaded: tmp/demographics_discovery/DS_RP_EMPLOI_LR_COMP.zip
Bytes: 17742302


In [51]:
with unemployment_download_path.open("rb") as file:
    unemployment_signature = file.read(4)

print("Signature:", unemployment_signature)
print(
    "Valid ZIP:",
    zipfile.is_zipfile(unemployment_download_path),
)

if unemployment_signature != b"PK\x03\x04":
    raise ValueError(
        "Unexpected download signature: "
        f"{unemployment_signature!r}"
    )

if not zipfile.is_zipfile(unemployment_download_path):
    raise zipfile.BadZipFile(
        f"Invalid ZIP archive: {unemployment_download_path}"
    )

Signature: b'PK\x03\x04'
Valid ZIP: True


In [53]:
with zipfile.ZipFile(unemployment_download_path) as archive:
    unemployment_archive_names = archive.namelist()
    unemployment_corrupt_member = archive.testzip()

print("Members:", unemployment_archive_names)
print("Corrupt member:", unemployment_corrupt_member)

if unemployment_corrupt_member is not None:
    raise zipfile.BadZipFile(
        "Corrupt archive member: "
        f"{unemployment_corrupt_member}"
    )

Members: ['DS_RP_EMPLOI_LR_COMP_2023_data.csv', 'DS_RP_EMPLOI_LR_COMP_2023_metadata.csv']
Corrupt member: None


In [54]:
unemployment_csv_names = [
    name
    for name in unemployment_archive_names
    if name.lower().endswith(".csv")
]

unemployment_data_names = [
    name
    for name in unemployment_csv_names
    if name.lower().endswith("_data.csv")
]

unemployment_metadata_names = [
    name
    for name in unemployment_csv_names
    if name.lower().endswith("_metadata.csv")
]

if len(unemployment_data_names) != 1:
    raise ValueError(
        "Expected one unemployment data CSV, found: "
        f"{unemployment_data_names}"
    )

if len(unemployment_metadata_names) != 1:
    raise ValueError(
        "Expected one unemployment metadata CSV, found: "
        f"{unemployment_metadata_names}"
    )

unemployment_data_name = unemployment_data_names[0]
unemployment_metadata_name = unemployment_metadata_names[0]

print("Data:", unemployment_data_name)
print("Metadata:", unemployment_metadata_name)

Data: DS_RP_EMPLOI_LR_COMP_2023_data.csv
Metadata: DS_RP_EMPLOI_LR_COMP_2023_metadata.csv


In [55]:
with zipfile.ZipFile(unemployment_download_path) as archive:
    with archive.open(unemployment_data_name) as file:
        unemployment_department_chunks = []

        for chunk in pd.read_csv(
            file,
            sep=";",
            dtype=str,
            chunksize=50_000,
            low_memory=False,
        ):
            if "GEO_OBJECT" not in chunk.columns:
                raise ValueError(
                    "The unemployment data has no "
                    "GEO_OBJECT column."
                )

            unemployment_department_chunks.append(
                chunk.loc[
                    chunk["GEO_OBJECT"].eq("DEP")
                ].copy()
            )

        if not unemployment_department_chunks:
            raise ValueError(
                "No departmental unemployment observations "
                "were found."
            )

        unemployment_raw = pd.concat(
            unemployment_department_chunks,
            ignore_index=True,
        )

    with archive.open(unemployment_metadata_name) as file:
        unemployment_codebook = pd.read_csv(
            file,
            sep=";",
            dtype=str,
        )

print(
    "Data:",
    unemployment_data_name,
    unemployment_raw.shape,
)

print(
    "Geographic levels retained:",
    unemployment_raw["GEO_OBJECT"].unique().tolist(),
)

print(
    "Metadata:",
    unemployment_metadata_name,
    unemployment_codebook.shape,
)

Data: DS_RP_EMPLOI_LR_COMP_2023_data.csv (6300, 10)
Geographic levels retained: ['DEP']
Metadata: DS_RP_EMPLOI_LR_COMP_2023_metadata.csv (41799, 5)


In [59]:
unemployment_raw.head(10)

,GEO,GEO_OBJECT,EMPSTA_ENQ,AGE,PCS,RP_MEASURE,FREQ,OBS_STATUS,TIME_PERIOD,OBS_VALUE
0,01,DEP,1,Y15T64,1,POP,A,A,2012,3661.8816
1,01,DEP,1,Y15T64,2,POP,A,A,2012,17245.90939
2,01,DEP,1,Y15T64,3,POP,A,A,2012,39209.87833
3,01,DEP,1,Y15T64,4,POP,A,A,2012,72168.85852
4,01,DEP,1,Y15T64,5,POP,A,A,2012,71133.35367
5,01,DEP,1,Y15T64,6,POP,A,A,2012,67581.92742
6,01,DEP,1,Y15T64,_T,POP,A,A,2012,271001.80893
7,01,DEP,1T2,Y15T64,1,POP,A,A,2012,3703.53869
8,01,DEP,1T2,Y15T64,2,POP,A,A,2012,18050.31902
9,01,DEP,1T2,Y15T64,3,POP,A,A,2012,40628.05507


In [57]:
unemployment_codebook.head(25)

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,GEO_OBJECT
0,AGE,Âge,Y15T64,De 15 à 64 ans,NaN
1,EMPSTA_ENQ,Statut d'emploi variable selon l'enquête,1T2,Actif,NaN
2,EMPSTA_ENQ,Statut d'emploi variable selon l'enquête,1,Actif occupé,NaN
3,EMPSTA_ENQ,Statut d'emploi variable selon l'enquête,2,Chômeur,NaN
4,FREQ,Fréquence,A,Annuel,NaN
5,OBS_STATUS,Statut de l'observation,K,Données inclues dans une autre catégorie,NaN
6,OBS_STATUS,Statut de l'observation,A,Normale,NaN
7,OBS_STATUS,Statut de l'observation,W,Inclut les données d’une autre catégorie,NaN
8,PCS,Profession et catégorie socioprofessionnelle (...,5,Employés,NaN
9,PCS,Profession et catégorie socioprofessionnelle (...,6,Ouvriers,NaN


In [58]:
inspect_dimensions(unemployment_raw)

,column,n_unique,sample_values
0,GEO,100,"[01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 1..."
1,GEO_OBJECT,1,[DEP]
2,EMPSTA_ENQ,3,"[1, 1T2, 2]"
3,AGE,1,[Y15T64]
4,PCS,7,"[1, 2, 3, 4, 5, 6, _T]"
5,RP_MEASURE,1,[POP]
6,FREQ,1,[A]
7,OBS_STATUS,1,[A]
8,TIME_PERIOD,3,"[2012, 2017, 2023]"
9,OBS_VALUE,6297,"[3661.8816, 17245.90939, 39209.87833, 72168.85..."


In [69]:
empsta_metadata = unemployment_codebook.loc[
    unemployment_codebook.astype(str).apply(
        lambda column: column.str.contains(
            r"EMPSTA_ENQ|1T2",
            case=False,
            regex=True,
            na=False,
        )
    ).any(axis=1)
].copy()

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(empsta_metadata)

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD,GEO_OBJECT
1,EMPSTA_ENQ,Statut d'emploi variable selon l'enquête,1T2,Actif,NaN
2,EMPSTA_ENQ,Statut d'emploi variable selon l'enquête,1,Actif occupé,NaN
3,EMPSTA_ENQ,Statut d'emploi variable selon l'enquête,2,Chômeur,NaN


In [70]:
unemployment_raw.loc[
    unemployment_raw["GEO"].eq("01")
    & unemployment_raw["PCS"].eq("_T"),
    [
        "GEO",
        "EMPSTA_ENQ",
        "AGE",
        "PCS",
        "TIME_PERIOD",
        "OBS_VALUE",
    ],
].sort_values(
    ["TIME_PERIOD", "EMPSTA_ENQ"],
    kind="stable",
)

,GEO,EMPSTA_ENQ,AGE,PCS,TIME_PERIOD,OBS_VALUE
6,01,1,Y15T64,_T,2012,271001.80893
13,01,1T2,Y15T64,_T,2012,298402.87484
20,01,2,Y15T64,_T,2012,27401.06591
27,01,1,Y15T64,_T,2017,282092.87382
34,01,1T2,Y15T64,_T,2017,312067.85555
41,01,2,Y15T64,_T,2017,29974.98173
48,01,1,Y15T64,_T,2023,306353.92376
55,01,1T2,Y15T64,_T,2023,332062.20638
62,01,2,Y15T64,_T,2023,25708.28261


In [71]:
employment_status_totals = (
    unemployment_raw.loc[
        unemployment_raw["PCS"].eq("_T")
    ]
    .pivot(
        index=["GEO", "TIME_PERIOD"],
        columns="EMPSTA_ENQ",
        values="OBS_VALUE",
    )
    .reset_index()
)

for column in ("1", "1T2", "2"):
    employment_status_totals[column] = pd.to_numeric(
        employment_status_totals[column],
        errors="coerce",
    )

employment_status_totals["active_balance"] = (
    employment_status_totals["1"]
    + employment_status_totals["2"]
    - employment_status_totals["1T2"]
)

employment_status_totals[
    [
        "active_balance",
    ]
].describe()

EMPSTA_ENQ,active_balance
count,3.000000e+02
mean,-2.999994e-07
std,4.931976e-06
min,-1.000002e-05
25%,0.000000e+00
50%,0.000000e+00
75%,0.000000e+00
max,1.000008e-05


In [72]:
employment_status_totals.assign(
    absolute_balance=lambda df: df["active_balance"].abs()
).sort_values(
    "absolute_balance",
    ascending=False,
).head(20)

EMPSTA_ENQ,GEO,TIME_PERIOD,1,1T2,2,active_balance,absolute_balance
286,95,2017,517554.50409,585550.02756,67995.52348,0.00001,0.00001
232,77,2017,615587.98571,686079.01830,70491.03260,0.00001,0.00001
233,77,2023,655671.06261,718659.37382,62988.31122,0.00001,0.00001
90,30,2012,263452.09417,311568.94762,48116.85346,0.00001,0.00001
128,42,2023,305542.57881,337807.37335,32264.79453,-0.00001,0.00001
67,24,2017,149095.40765,171960.30917,22864.90153,0.00001,0.00001
151,50,2017,195391.44418,217038.44408,21646.99991,0.00001,0.00001
92,30,2023,286965.34708,326403.52114,39438.17405,-0.00001,0.00001
160,53,2017,125757.57045,137574.69291,11817.12245,-0.00001,0.00001
70,25,2017,219880.54953,247781.57801,27901.02847,-0.00001,0.00001


In [73]:
employment_status_totals["unemployment_rate_percent"] = (
    employment_status_totals["2"]
    / employment_status_totals["1T2"]
    * 100
)

employment_status_totals[
    "unemployment_rate_percent"
].describe()

count    300.000000
mean      11.401607
std        3.020544
min        5.855735
25%        9.517807
50%       11.095293
75%       12.492775
max       29.672798
Name: unemployment_rate_percent, dtype: float64

In [76]:
employment_status_totals["TIME_PERIOD"] = pd.to_numeric(
    employment_status_totals["TIME_PERIOD"],
    errors="raise",
)

employment_status_totals[
    "census_unemployment_rate_15_64_percent"
] = (
    employment_status_totals["2"]
    / employment_status_totals["1T2"]
    * 100
)

latest_unemployment_year = (
    employment_status_totals["TIME_PERIOD"].max()
)

unemployment_rate = (
    employment_status_totals.loc[
        employment_status_totals["TIME_PERIOD"].eq(
            latest_unemployment_year
        ),
        [
            "GEO",
            "TIME_PERIOD",
            "1",
            "2",
            "1T2",
            "census_unemployment_rate_15_64_percent",
        ],
    ]
    .rename(
        columns={
            "1": "employed_population_15_64",
            "2": "unemployed_population_15_64",
            "1T2": "active_population_15_64",
        }
    )
    .sort_values("GEO", kind="stable")
    .reset_index(drop=True)
)

In [77]:
print("Latest year:", latest_unemployment_year)
print("Rows:", len(unemployment_rate))
print("Departments:", unemployment_rate["GEO"].nunique())

print(
    unemployment_rate[
        "census_unemployment_rate_15_64_percent"
    ].describe()
)

unemployment_rate.head()

Latest year: 2023
Rows: 100
Departments: 100
count    100.000000
mean       9.662308
std        2.192046
min        5.855735
25%        8.546830
50%        9.298903
75%       10.169298
max       21.272933
Name: census_unemployment_rate_15_64_percent, dtype: float64


EMPSTA_ENQ,GEO,TIME_PERIOD,employed_population_15_64,unemployed_population_15_64,active_population_15_64,census_unemployment_rate_15_64_percent
0,01,2023,306353.92376,25708.28261,332062.20638,7.742008
1,02,2023,198151.87740,28886.66608,227038.54348,12.723243
2,03,2023,124010.40923,14239.40570,138249.81493,10.299765
3,04,2023,64198.24881,7021.45055,71219.69936,9.858860
4,05,2023,59286.34693,4869.77287,64156.11979,7.590504


In [79]:
employment_status_totals["active_balance"] = (
    employment_status_totals["1"]
    + employment_status_totals["2"]
    - employment_status_totals["1T2"]
)

max_balance = (
    employment_status_totals["active_balance"]
    .abs()
    .max()
)

if employment_status_totals["1T2"].le(0).any():
    raise ValueError(
        "Active population contains zero or negative values."
    )

if max_balance > 0.01:
    raise ValueError(
        "Employment-status components do not reconcile: "
        f"maximum absolute balance is {max_balance}."
    )

In [81]:
unemployment_rate.sort_values(
    "census_unemployment_rate_15_64_percent",
    ascending=False,
).head(10)

EMPSTA_ENQ,GEO,TIME_PERIOD,employed_population_15_64,unemployed_population_15_64,active_population_15_64,census_unemployment_rate_15_64_percent
99,974,2023,296040.56925,79993.46835,376034.03759,21.272933
96,971,2023,129106.24997,28262.63834,157368.88831,17.959483
66,66,2023,169449.66010,29356.83490,198806.49500,14.766537
97,972,2023,130747.28533,21913.85789,152661.14323,14.354575
98,973,2023,79257.54000,12814.10531,92071.64531,13.917537
10,11,2023,136034.60058,20647.31267,156681.91325,13.177853
1,02,2023,198151.87740,28886.66608,227038.54348,12.723243
34,34,2023,474910.29541,68471.55172,543381.84713,12.601001
7,08,2023,101721.21760,14116.34636,115837.56396,12.186329
30,30,2023,286965.34708,39438.17405,326403.52114,12.082644


## Population data

Search the departmental Melodi catalogue for 2023 population and density products. Population density may be published directly or may need to be derived from population and area, so inspect both dataset titles and measure metadata before selecting a source.


In [83]:
population_search_terms = (
    "population",
    "population municipale",
    "densité",
    "superficie",
)

In [84]:
population_candidate_frames = []

for term in population_search_terms:
    matches = search_catalog(term).copy()
    matches.insert(0, "search_term", term)
    population_candidate_frames.append(matches)

population_candidates = (
    pd.concat(population_candidate_frames, ignore_index=True)
    .drop_duplicates("identifier")
    .sort_values("identifier", kind="stable")
    .reset_index(drop=True)
)

population_candidates

,search_term,identifier,title,start_period,end_period,spatial_levels,num_observations
0,population,DS_ESTIMATION_POPULATION,Estimations localisées de population - séries ...,1975-01-01T00:00:00,2026-01-01T00:00:00,"[OTHER, DEP, FRANCE, REG]",432882
1,population,DS_POPULATIONS_HISTORIQUES,Populations municipales de 1968 à 2023,1968-01-01T00:00:00,2023-01-01T00:00:00,"[ARR, ARM, COM, DEP, REG, FRANCE]",813234
2,population,DS_POPULATIONS_REFERENCE,Populations de référence,2023-01-01T00:00:00,2023-01-01T00:00:00,"[COM, ARM, ARR, DEP, REG, FRANCE]",106065
3,population,DS_RP_ACTIVITE_PRINC,Caractéristiques de l'emploi - Principaux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",8646183
4,population,DS_RP_DIPLOMES_PRINC,Diplômes et Formation – Principaux indicateurs...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",3383289
5,population,DS_RP_EDUCATION,Éducation - Scolarisation,2011-01-01T00:00:00,2022-01-01T00:00:00,"[COM, ARR, DEP, REG, EPCI, AAV2020, UU2020, ZE...",7894530
6,population,DS_RP_EDUCATION_PRINC,Éducation - Scolarisation – Principaux indicat...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, FRANCE, ...",9022104
7,population,DS_RP_EMPLOI_LR_COMP,Population active et chômage - Princiapux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",2631447
8,population,DS_RP_EMPLOI_LR_PRINC,Population active et chômage - Principaux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",16039296
9,population,DS_RP_EMPLOI_LT_PRINC,Emploi au lieu de travail - Principaux indicat...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",2004912


In [85]:
population_candidates_2023 = population_candidates.loc[
    pd.to_datetime(
        population_candidates["start_period"],
        errors="coerce",
    ).dt.year.le(2023)
    & pd.to_datetime(
        population_candidates["end_period"],
        errors="coerce",
    ).dt.year.ge(2023)
].copy()
population_candidates_2023

,search_term,identifier,title,start_period,end_period,spatial_levels,num_observations
0,population,DS_ESTIMATION_POPULATION,Estimations localisées de population - séries ...,1975-01-01T00:00:00,2026-01-01T00:00:00,"[OTHER, DEP, FRANCE, REG]",432882
1,population,DS_POPULATIONS_HISTORIQUES,Populations municipales de 1968 à 2023,1968-01-01T00:00:00,2023-01-01T00:00:00,"[ARR, ARM, COM, DEP, REG, FRANCE]",813234
2,population,DS_POPULATIONS_REFERENCE,Populations de référence,2023-01-01T00:00:00,2023-01-01T00:00:00,"[COM, ARM, ARR, DEP, REG, FRANCE]",106065
3,population,DS_RP_ACTIVITE_PRINC,Caractéristiques de l'emploi - Principaux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",8646183
4,population,DS_RP_DIPLOMES_PRINC,Diplômes et Formation – Principaux indicateurs...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",3383289
6,population,DS_RP_EDUCATION_PRINC,Éducation - Scolarisation – Principaux indicat...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, FRANCE, ...",9022104
7,population,DS_RP_EMPLOI_LR_COMP,Population active et chômage - Princiapux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",2631447
8,population,DS_RP_EMPLOI_LR_PRINC,Population active et chômage - Principaux indi...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",16039296
9,population,DS_RP_EMPLOI_LT_PRINC,Emploi au lieu de travail - Principaux indicat...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",2004912
10,population,DS_RP_MENAGES_PRINC,Ménages et couple – Principaux indicateurs (do...,2012-01-01T00:00:00,2023-01-01T00:00:00,"[AAV2020, ARR, ARM, BV2022, COM, DEP, EPCI, FR...",4552821


In [89]:
population_dataset_id = "DS_POPULATIONS_REFERENCE"

population_dataset_metadata = get_dataset_catalog(
    population_dataset_id
)

population_dataset_metadata.keys()

dict_keys(['abstract', 'accessRights', 'accrualPeriodicity', 'confidentialityStatus', 'creator', 'description', 'identifier', 'issued', 'keyword', 'landingPage', 'modified', 'numObservations', 'numSeries', 'processStep', 'product', 'publisher', 'relations', 'scopeNote', 'spatial', 'spatialResolution', 'spatialTemporal', 'statisticalUnit', 'status', 'structure', 'subtitle', 'temporal', 'temporalResolution', 'theme', 'title', 'type', 'uri', 'uuid', 'wasGeneratedBy'])

In [90]:
population_csv_products = [
    product
    for product in population_dataset_metadata.get("product", [])
    if product.get("format") == "CSV"
]

pd.DataFrame(population_csv_products)

,accessURL,byteSize,format,id,issued,language,mediaType,modified,packageFormat,title
0,https://api.insee.fr/melodi/file/DS_POPULATION...,985016,CSV,DS_POPULATIONS_REFERENCE_2023_CSV_FR,2025-12-17T08:50:56,FR,text/csv,2026-02-17T12:08:49.311695357,application/zip,Populations de référence


In [92]:
if len(population_csv_products) != 1:
    raise ValueError(
        f"Expected one CSV product, found {len(population_csv_products)}"
    )

population_csv_product = population_csv_products[0]
population_csv_url = population_csv_product["accessURL"]

population_csv_url

'https://api.insee.fr/melodi/file/DS_POPULATIONS_REFERENCE/DS_POPULATIONS_REFERENCE_2023_CSV_FR'

In [93]:
population_download_path = (
    discovery_dir / f"{population_dataset_id}.zip"
)

population_response = requests.get(
    population_csv_url,
    timeout=(TIMEOUT, 300),
)
population_response.raise_for_status()

population_download_path.write_bytes(
    population_response.content
)

print("Downloaded:", population_download_path)
print("Bytes:", population_download_path.stat().st_size)
print("Valid ZIP:", zipfile.is_zipfile(population_download_path))

Downloaded: tmp/demographics_discovery/DS_POPULATIONS_REFERENCE.zip
Bytes: 985016
Valid ZIP: True


In [94]:
with zipfile.ZipFile(population_download_path) as archive:
    population_archive_names = archive.namelist()

population_archive_names

['DS_POPULATIONS_REFERENCE_2023_metadata.csv',
 'DS_POPULATIONS_REFERENCE_2023_data.csv']

In [95]:
population_data_names = [
    name
    for name in population_archive_names
    if name.lower().endswith("_data.csv")
]

population_metadata_names = [
    name
    for name in population_archive_names
    if name.lower().endswith("_metadata.csv")
]

print("Data:", population_data_names)
print("Metadata:", population_metadata_names)

Data: ['DS_POPULATIONS_REFERENCE_2023_data.csv']
Metadata: ['DS_POPULATIONS_REFERENCE_2023_metadata.csv']


In [96]:
if len(population_data_names) != 1:
    raise ValueError(
        f"Expected one data CSV, found: {population_data_names}"
    )

if len(population_metadata_names) != 1:
    raise ValueError(
        f"Expected one metadata CSV, found: {population_metadata_names}"
    )

with zipfile.ZipFile(population_download_path) as archive:
    with archive.open(population_data_names[0]) as file:
        department_chunks = []

        for chunk in pd.read_csv(
            file,
            sep=";",
            dtype=str,
            chunksize=50_000,
            low_memory=False,
        ):
            department_chunks.append(
                chunk.loc[
                    chunk["GEO_OBJECT"].eq("DEP")
                ].copy()
            )

        population_raw = pd.concat(
            department_chunks,
            ignore_index=True,
        )

    with archive.open(population_metadata_names[0]) as file:
        population_codebook = pd.read_csv(
            file,
            sep=";",
            dtype=str,
        )

print("Data shape:", population_raw.shape)
print("Metadata shape:", population_codebook.shape)

Data shape: (300, 6)
Metadata shape: (35360, 4)


In [97]:
inspect_dimensions(population_raw)

,column,n_unique,sample_values
0,GEO,100,"[18, 26, 39, 49, 971, 43, 972, 75, 41, 33, 35,..."
1,GEO_OBJECT,1,[DEP]
2,FREQ,1,[A]
3,POPREF_MEASURE,3,"[PCAP, PTOT, PMUN]"
4,TIME_PERIOD,1,[2023]
5,OBS_VALUE,300,"[6573, 12741, 266940, 19992, 384160, 6921, 360..."


In [98]:
population_measure_metadata = population_codebook.loc[
    population_codebook.astype(str).apply(
        lambda column: column.str.contains(
            r"POPREF_MEASURE|PMUN|PCAP|PTOT",
            case=False,
            regex=True,
            na=False,
        )
    ).any(axis=1)
].copy()

with pd.option_context(
    "display.max_rows", 100,
    "display.max_columns", None,
    "display.max_colwidth", None,
):
    display(population_measure_metadata)

,COD_VAR,LIB_VAR,COD_MOD,LIB_MOD
1,POPREF_MEASURE,Mesure,PMUN,Population municipale
2,POPREF_MEASURE,Mesure,PCAP,Population comptée à part
3,POPREF_MEASURE,Mesure,PTOT,Population totale
26331,GEO,Géographie,27083,Bonneville-Aptot


In [99]:
population_components = (
    population_raw
    .pivot(
        index=["GEO", "TIME_PERIOD"],
        columns="POPREF_MEASURE",
        values="OBS_VALUE",
    )
    .reset_index()
)

for column in ("PMUN", "PCAP", "PTOT"):
    population_components[column] = pd.to_numeric(
        population_components[column],
        errors="raise",
    )

population_components.head()

POPREF_MEASURE,GEO,TIME_PERIOD,PCAP,PMUN,PTOT
0,01,2023,15601,679344,694945
1,02,2023,11038,523342,534380
2,03,2023,8415,333298,341713
3,04,2023,4331,168054,172385
4,05,2023,4289,143467,147756


In [101]:
population_components["total_balance"] = (
    population_components["PMUN"]
    + population_components["PCAP"]
    - population_components["PTOT"]
)

max_population_balance = (
    population_components["total_balance"]
    .abs()
    .max()
)

if max_population_balance != 0:
    raise ValueError(
        "Population components do not reconcile: "
        f"maximum balance is {max_population_balance}."
    )

In [102]:
population_2023 = (
    population_components.loc[
        population_components["TIME_PERIOD"].astype(int).eq(2023),
        [
            "GEO",
            "TIME_PERIOD",
            "PMUN",
            "PCAP",
            "PTOT",
        ],
    ]
    .rename(
        columns={
            "PMUN": "municipal_population",
            "PCAP": "population_counted_separately",
            "PTOT": "total_population",
        }
    )
    .sort_values("GEO", kind="stable")
    .reset_index(drop=True)
)

population_2023.head()

POPREF_MEASURE,GEO,TIME_PERIOD,municipal_population,population_counted_separately,total_population
0,01,2023,679344,15601,694945
1,02,2023,523342,11038,534380
2,03,2023,333298,8415,341713
3,04,2023,168054,4331,172385
4,05,2023,143467,4289,147756


## Departmental Area

In [115]:
import geopandas as gpd

departments_gdf = gpd.read_file(
    "data/raw/geodata/departments.geojson"
)

print(departments_gdf.crs)
print(departments_gdf.columns.tolist())
print(f"Shape: {departments_gdf.shape}")
departments_gdf.head()

EPSG:4326
['code', 'nom', 'geometry']
Shape: (96, 3)


,code,nom,geometry
0,01,Ain,"POLYGON ((4.78021 46.17668, 4.78024 46.18905, ..."
1,02,Aisne,"POLYGON ((3.17296 50.01131, 3.17382 50.01186, ..."
2,03,Allier,"POLYGON ((3.03207 46.79491, 3.03424 46.7908, 3..."
3,04,Alpes-de-Haute-Provence,"POLYGON ((5.67604 44.19143, 5.67817 44.19051, ..."
4,05,Hautes-Alpes,"POLYGON ((6.26057 45.12685, 6.26417 45.12641, ..."


In [116]:
departments_gdf["GEO"] = (
    departments_gdf["code"]
    .astype(str)
    .str.strip()
)

departments_gdf = departments_gdf[
    ["GEO", "nom", "geometry"]
]

In [117]:
departments_l93 = departments_gdf.to_crs("EPSG:2154")

print(departments_l93.crs)
print(departments_l93.columns.tolist())
print(f"Shape: {departments_l93.shape}")

EPSG:2154
['GEO', 'nom', 'geometry']
Shape: (96, 3)


In [118]:
department_area = (
    departments_l93[
        ["GEO", "nom", "geometry"]
    ]
    .assign(
        area_sq_km=lambda df:
            df.geometry.area / 1_000_000
    )
    .drop(columns="geometry")
    .sort_values("GEO", kind="stable")
    .reset_index(drop=True)
)

department_area.head()

,GEO,nom,area_sq_km
0,01,Ain,5774.887544
1,02,Aisne,7419.546299
2,03,Allier,7365.400369
3,04,Alpes-de-Haute-Provence,6994.235138
4,05,Hautes-Alpes,5685.756003


In [119]:
print("Rows:", len(department_area))
print("Departments:", department_area["GEO"].nunique())

department_area["area_sq_km"].describe()

Rows: 96
Departments: 96


count       96.000000
mean      5717.445028
std       1951.271295
min        105.345361
25%       5152.831563
50%       5993.122832
75%       6811.143680
max      10374.431284
Name: area_sq_km, dtype: float64

In [120]:
department_area.sort_values("area_sq_km").head(10)

department_area.sort_values(
    "area_sq_km",
    ascending=False,
).head(10)

,GEO,nom,area_sq_km
33,33,Gironde,10374.431284
40,40,Landes,9352.602563
22,24,Dordogne,9210.880031
19,21,Côte-d'Or,8787.360997
11,12,Aveyron,8770.220063
71,71,Saône-et-Loire,8598.567682
51,51,Marne,8196.602621
63,63,Puy-de-Dôme,8001.631590
38,38,Isère,7868.215105
64,64,Pyrénées-Atlantiques,7696.974601


## Derive population density

In [121]:
population_density_2023 = (
    population_2023.loc[
        population_2023["GEO"].isin(department_area["GEO"])
    ]
    .merge(
        department_area[
            ["GEO", "nom", "area_sq_km"]
        ],
        on="GEO",
        how="inner",
        validate="one_to_one",
    )
)

In [122]:
population_density_2023[
    "population_density_per_sq_km"
] = (
    population_density_2023["municipal_population"]
    / population_density_2023["area_sq_km"]
)

In [123]:
if len(population_density_2023) != 96:
    raise ValueError(
        "Expected 96 metropolitan departments, "
        f"found {len(population_density_2023)}."
    )

if population_density_2023["GEO"].nunique() != 96:
    raise ValueError("Department codes are not unique.")

if population_density_2023[
    ["municipal_population", "area_sq_km"]
].isna().any().any():
    raise ValueError(
        "Population or area is missing after the merge."
    )

if population_density_2023["area_sq_km"].le(0).any():
    raise ValueError(
        "Departmental area contains zero or negative values."
    )

In [124]:
population_density_2023[
    [
        "GEO",
        "nom",
        "TIME_PERIOD",
        "municipal_population",
        "area_sq_km",
        "population_density_per_sq_km",
    ]
].sort_values(
    "population_density_per_sq_km",
    ascending=False,
).head(15)

,GEO,nom,TIME_PERIOD,municipal_population,area_sq_km,population_density_per_sq_km
75,75,Paris,2023,2103778,105.345361,19970.295645
92,92,Hauts-de-Seine,2023,1654712,175.504294,9428.327718
93,93,Seine-Saint-Denis,2023,1704316,236.784539,7197.750362
94,94,Val-de-Marne,2023,1426929,244.895763,5826.678992
95,95,Val-d'Oise,2023,1281653,1253.224985,1022.683888
91,91,Essonne,2023,1338485,1818.450791,736.057861
78,78,Yvelines,2023,1485086,2306.187107,643.957290
69,69,Rhône,2023,1914667,3252.850943,588.611969
59,59,Nord,2023,2615635,5767.754470,453.492778
12,13,Bouches-du-Rhône,2023,2087658,5094.560716,409.781749


## GDP

In [138]:
from io import StringIO

OECD_GDP_API = (
    "https://sdmx.oecd.org/public/rest/data/"
    "OECD.CFE.EDS,DSD_REG_ECO@DF_GDP,2.4/all"
)

oecd_gdp_response = requests.get(
    OECD_GDP_API,
    params={
        "startPeriod": "2020",
        "dimensionAtObservation": "AllDimensions",
        "format": "csvfilewithlabels",
    },
    timeout=(30, 300),
)

oecd_gdp_response.raise_for_status()

oecd_gdp_raw = pd.read_csv(
    StringIO(oecd_gdp_response.text),
    dtype=str,
    low_memory=False,
)

print("Shape:", oecd_gdp_raw.shape)
print(oecd_gdp_raw.columns.tolist())

Shape: (171955, 36)
['STRUCTURE', 'STRUCTURE_ID', 'STRUCTURE_NAME', 'ACTION', 'FREQ', 'Frequency of observation', 'TERRITORIAL_LEVEL', 'Territorial level', 'REF_AREA', 'Reference area', 'TERRITORIAL_TYPE', 'Territorial typology', 'MEASURE', 'Measure', 'ACTIVITY', 'Economic activity', 'PRICES', 'Price base', 'UNIT_MEASURE', 'Unit of measure', 'TIME_PERIOD', 'Time period', 'OBS_VALUE', 'Observation value', 'COUNTRY', 'Country', 'OBS_STATUS', 'Observation status', 'REF_YEAR_PRICE', 'Price reference year', 'UNIT_MULT', 'Unit multiplier', 'DECIMALS', 'Decimals', 'CURRENCY', 'Currency']


In [139]:
for column in oecd_gdp_raw.columns:
    unique_values = oecd_gdp_raw[column].dropna().unique()

    if len(unique_values) <= 100:
        print(
            column,
            len(unique_values),
            unique_values[:100],
        )

STRUCTURE 1 ['DATAFLOW']
STRUCTURE_ID 1 ['OECD.CFE.EDS:DSD_REG_ECO@DF_GDP(2.4)']
STRUCTURE_NAME 1 ['Gross domestic product - Regions']
ACTION 1 ['I']
FREQ 1 ['A']
Frequency of observation 1 ['Annual']
TERRITORIAL_LEVEL 3 ['TL3' 'TL2' 'CTRY']
Territorial level 3 ['TL3' 'TL2' 'Country']
TERRITORIAL_TYPE 1 ['_Z']
Territorial typology 1 ['Not applicable']
MEASURE 4 ['GDP' 'PPP_GDP' 'POP' 'EMP']
Measure 4 ['Gross domestic product' 'Purchasing Power Parities for GDP' 'Population'
 'Employment']
ACTIVITY 1 ['_T']
Economic activity 1 ['Total - all activities']
PRICES 4 ['V' 'Q' '_Z' 'D']
Price base 4 ['Current prices' 'Constant prices' 'Not applicable' 'Deflator']
UNIT_MEASURE 11 ['USD_PPP_WR' 'USD_PPP_PS' 'USD_PPP' 'USD_PS' 'USD' 'XDC_PS' 'XDC_WR'
 'XDC' 'XDC_USD' 'IX' 'PS']
Unit of measure 11 ['US dollars per worker, PPP converted'
 'US dollars per person, PPP converted' 'US dollars, PPP converted'
 'US dollars per person' 'US dollar' 'National currency per person'
 'National currency per wo

In [140]:
oecd_france_tl3 = oecd_gdp_raw.loc[
    oecd_gdp_raw["COUNTRY"].eq("FRA")
    & oecd_gdp_raw["TERRITORIAL_LEVEL"].eq("TL3")
].copy()

print("Rows:", len(oecd_france_tl3))
print(
    "TL3 territories:",
    oecd_france_tl3["REF_AREA"].nunique(),
)

oecd_france_tl3[
    [
        "REF_AREA",
        "Reference area",
        "TIME_PERIOD",
        "MEASURE",
        "PRICES",
        "UNIT_MEASURE",
        "UNIT_MULT",
        "CURRENCY",
        "OBS_STATUS",
    ]
].drop_duplicates().head(30)

Rows: 8206
TL3 territories: 102


,REF_AREA,Reference area,TIME_PERIOD,MEASURE,PRICES,UNIT_MEASURE,UNIT_MULT,CURRENCY,OBS_STATUS
11820,FRL02,Hautes-Alpes,2021,GDP,V,USD_PPP_WR,0,USD,A
11821,FRK13,Haute-Loire,2021,GDP,V,USD_PPP_WR,0,USD,A
11822,FRB03,Indre,2021,GDP,V,USD_PPP_WR,0,USD,A
11823,FRM01,Corse-du-Sud,2021,GDP,V,USD_PPP_WR,0,USD,A
11824,FRI14,Lot-et-Garonne,2021,GDP,V,USD_PPP_WR,0,USD,A
11825,FRK11,Allier,2021,GDP,V,USD_PPP_WR,0,USD,A
11826,FRH03,Ille-et-Vilaine,2021,GDP,V,USD_PPP_WR,0,USD,A
11827,FRF34,Vosges,2021,GDP,V,USD_PPP_WR,0,USD,A
11828,FRK24,Isère,2021,GDP,V,USD_PPP_WR,0,USD,A
11829,FR107,Val-de-Marne,2021,GDP,V,USD_PPP_WR,0,USD,A


In [141]:
oecd_france_tl3_areas = (
    oecd_france_tl3[
        ["REF_AREA", "Reference area"]
    ]
    .drop_duplicates()
    .sort_values("REF_AREA", kind="stable")
    .reset_index(drop=True)
)

print("French TL3 areas:", len(oecd_france_tl3_areas))

with pd.option_context(
    "display.max_rows", 200,
    "display.max_colwidth", None,
):
    display(oecd_france_tl3_areas)

French TL3 areas: 102


,REF_AREA,Reference area
0,FR101,Paris
1,FR102,Seine-et-Marne
2,FR103,Yvelines
3,FR104,Essonne
4,FR105,Hauts-de-Seine
5,FR106,Seine-Saint-Denis
6,FR107,Val-de-Marne
7,FR108,Val-d'Oise
8,FRB01,Cher
9,FRB02,Eure-et-Loir


In [142]:
oecd_gdp_2023 = oecd_france_tl3.loc[
    oecd_france_tl3["TIME_PERIOD"].eq("2023")
    & oecd_france_tl3["MEASURE"].eq("GDP")
    & oecd_france_tl3["ACTIVITY"].eq("_T")
    & oecd_france_tl3["PRICES"].eq("V")
    & oecd_france_tl3["UNIT_MEASURE"].eq("XDC")
    & oecd_france_tl3["UNIT_MULT"].eq("6")
    & oecd_france_tl3["CURRENCY"].eq("EUR")
].copy()

print("Rows:", len(oecd_gdp_2023))
print("Territories:", oecd_gdp_2023["REF_AREA"].nunique())

oecd_gdp_2023[
    [
        "REF_AREA",
        "Reference area",
        "TIME_PERIOD",
        "OBS_VALUE",
        "OBS_STATUS",
        "Observation status",
    ]
].head(20)

Rows: 102
Territories: 102


,REF_AREA,Reference area,TIME_PERIOD,OBS_VALUE,OBS_STATUS,Observation status
96482,FRL06,Vaucluse,2023,20710.1,P,Provisional value
96483,FRJ22,Aveyron,2023,8544.7,P,Provisional value
96484,FRD12,Manche,2023,16965.5,P,Provisional value
96485,FRE22,Oise,2023,25558.3,P,Provisional value
96486,FRH03,Ille-et-Vilaine,2023,45322.9,P,Provisional value
96487,FRY20,Martinique,2023,10098.2,P,Provisional value
96488,FRI33,Deux-Sèvres,2023,12807,P,Provisional value
96489,FRK26,Rhône,2023,107089.4,P,Provisional value
96490,FRG05,Vendée,2023,25116,P,Provisional value
96491,FRB02,Eure-et-Loir,2023,13312.7,P,Provisional value


In [143]:
duplicate_gdp_rows = oecd_gdp_2023.duplicated(
    subset=["REF_AREA", "TIME_PERIOD"],
    keep=False,
)

print("Duplicate territory-year rows:", duplicate_gdp_rows.sum())

print(
    oecd_gdp_2023[
        ["OBS_STATUS", "Observation status"]
    ]
    .value_counts(dropna=False)
)

Duplicate territory-year rows: 0
OBS_STATUS  Observation status
P           Provisional value     102
Name: count, dtype: int64


In [144]:
oecd_france_tl3_gdp_coverage = (
    oecd_france_tl3.loc[
        oecd_france_tl3["MEASURE"].eq("GDP")
        & oecd_france_tl3["ACTIVITY"].eq("_T")
        & oecd_france_tl3["PRICES"].eq("V")
        & oecd_france_tl3["UNIT_MEASURE"].eq("XDC")
        & oecd_france_tl3["UNIT_MULT"].eq("6")
        & oecd_france_tl3["CURRENCY"].eq("EUR")
    ]
    .groupby("TIME_PERIOD", as_index=False)
    .agg(
        rows=("OBS_VALUE", "size"),
        territories=("REF_AREA", "nunique"),
        missing_values=("OBS_VALUE", lambda values: values.isna().sum()),
    )
    .sort_values("TIME_PERIOD")
)

oecd_france_tl3_gdp_coverage

,TIME_PERIOD,rows,territories,missing_values
0,2020,102,102,0
1,2021,102,102,0
2,2022,102,102,0
3,2023,102,102,0
4,2024,102,102,0


### Normalise names

In [145]:
import re
import unicodedata


def normalize_department_name(value: str) -> str:
    value = unicodedata.normalize("NFKD", value)
    value = "".join(
        character
        for character in value
        if not unicodedata.combining(character)
    )
    value = value.casefold()
    value = value.replace("’", "'")
    value = re.sub(r"[^a-z0-9]+", " ", value)
    return " ".join(value.split())

In [146]:
oecd_gdp_2023_clean = (
    oecd_gdp_2023[
        [
            "REF_AREA",
            "Reference area",
            "TIME_PERIOD",
            "OBS_VALUE",
            "OBS_STATUS",
            "Observation status",
        ]
    ]
    .rename(
        columns={
            "REF_AREA": "oecd_tl3_code",
            "Reference area": "oecd_department_name",
            "OBS_VALUE": "gdp_current_prices_million_eur",
            "OBS_STATUS": "gdp_observation_status",
            "Observation status": "gdp_observation_status_label",
        }
    )
    .assign(
        department_name_key=lambda df:
            df["oecd_department_name"].map(
                normalize_department_name
            ),
        gdp_current_prices_million_eur=lambda df:
            pd.to_numeric(
                df["gdp_current_prices_million_eur"],
                errors="raise",
            ),
    )
)

In [147]:
oecd_gdp_2023_clean

,oecd_tl3_code,oecd_department_name,TIME_PERIOD,gdp_current_prices_million_eur,gdp_observation_status,gdp_observation_status_label,department_name_key
96482,FRL06,Vaucluse,2023,20710.1,P,Provisional value,vaucluse
96483,FRJ22,Aveyron,2023,8544.7,P,Provisional value,aveyron
96484,FRD12,Manche,2023,16965.5,P,Provisional value,manche
96485,FRE22,Oise,2023,25558.3,P,Provisional value,oise
96486,FRH03,Ille-et-Vilaine,2023,45322.9,P,Provisional value,ille et vilaine
...,...,...,...,...,...,...,...
96579,FR104,Essonne,2023,62004.0,P,Provisional value,essonne
96580,FRF11,Bas-Rhin,2023,46884.9,P,Provisional value,bas rhin
96581,FRB06,Loiret,2023,26301.8,P,Provisional value,loiret
96582,FRI12,Gironde,2023,70556.3,P,Provisional value,gironde


### Attempting merge

In [148]:
department_crosswalk = (
    department_area[
        ["GEO", "nom"]
    ]
    .rename(columns={"nom": "department_name"})
    .assign(
        department_name_key=lambda df:
            df["department_name"].map(
                normalize_department_name
            )
    )
)

In [149]:
gdp_department_match = oecd_gdp_2023_clean.merge(
    department_crosswalk,
    on="department_name_key",
    how="left",
    validate="many_to_one",
)

unmatched_oecd_areas = (
    gdp_department_match.loc[
        gdp_department_match["GEO"].isna(),
        [
            "oecd_tl3_code",
            "oecd_department_name",
        ],
    ]
    .drop_duplicates()
    .sort_values("oecd_department_name")
)

unmatched_oecd_areas

,oecd_tl3_code,oecd_department_name
101,FRZZZ,"France, not regionalised"
52,FRY30,French Guiana
53,FRY10,Guadeloupe
15,FRY40,La Réunion
5,FRY20,Martinique
59,FRY50,Mayotte


In [150]:
matched_metropolitan_gdp = (
    gdp_department_match.loc[
        gdp_department_match["GEO"].notna()
    ]
    .copy()
)

print("Matched rows:", len(matched_metropolitan_gdp))
print(
    "Matched departments:",
    matched_metropolitan_gdp["GEO"].nunique(),
)

Matched rows: 96
Matched departments: 96


In [151]:
matched_metropolitan_gdp

,oecd_tl3_code,oecd_department_name,TIME_PERIOD,gdp_current_prices_million_eur,gdp_observation_status,gdp_observation_status_label,department_name_key,GEO,department_name
0,FRL06,Vaucluse,2023,20710.1,P,Provisional value,vaucluse,84,Vaucluse
1,FRJ22,Aveyron,2023,8544.7,P,Provisional value,aveyron,12,Aveyron
2,FRD12,Manche,2023,16965.5,P,Provisional value,manche,50,Manche
3,FRE22,Oise,2023,25558.3,P,Provisional value,oise,60,Oise
4,FRH03,Ille-et-Vilaine,2023,45322.9,P,Provisional value,ille et vilaine,35,Ille-et-Vilaine
...,...,...,...,...,...,...,...,...,...
96,FRK23,Drôme,2023,21052.3,P,Provisional value,drome,26,Drôme
97,FR104,Essonne,2023,62004.0,P,Provisional value,essonne,91,Essonne
98,FRF11,Bas-Rhin,2023,46884.9,P,Provisional value,bas rhin,67,Bas-Rhin
99,FRB06,Loiret,2023,26301.8,P,Provisional value,loiret,45,Loiret
